# Lab 8 — Memory-Backed Agent + Contradiction Detector
### *Add memory to an agent, then test how contradiction handling changes outcomes.*

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/labs/agents2_lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Overview
This lab builds on the lecture TinyMemory starter and adds:
- memory writes (`remember`)
- memory retrieval (`search`)
- memory deletion (`forget`)
- contradiction detection with DSPy `Predict`

Small experiment:
> Does contradiction detection improve factual accuracy after memory updates?

---

## Learning goals
- Use the lecture starter `TinyMemory` implementation.
- Add the `forget` behavior for removing stale memories.
- Build a contradiction detector using DSPy `Predict`.
- Implement a tiny memory-aware agent loop with contradiction checks.
- Run a controlled experiment: detector OFF vs ON.


In [1]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append("/content/main")

import re, json
import numpy as np
import pandas as pd

from course_utils import lab8_setup, get_text_embedding

lab8_setup()
import dspy
print("✅ Environment ready! dspy =", "yes" if dspy else "no")


🔧 Setting up your environment...
  → Installing core packages...
installing mermaid-python
  → Installing additional packages: dspy
installing dspy
  → Setting random seed for reproducible results...
  → Checking API key...
🔑 Enter your OpenAI API key.
   (It will only be stored in this Colab runtime - it's safe!)
   Get your key from: https://platform.openai.com/api-keys
OpenAI API key: ··········
✅ API key set.
  → Adding course files to path...
✅ Setup complete!
✅ lab8_setup complete — ready.
✅ Environment ready! dspy = yes


--- **TODO**

## Pre-Lab Questions
Answer in 1–2 sentences each. (Edit this cell.)

1. What is one benefit of long-term memory in an agent?
2. What is one risk of long-term memory?
3. Why might contradictory memories degrade an agent's answers?

**Your answers:**
1)  
2)  
3)


--- **TODO**

## Scientific Question & Hypothesis

**Question:**  
If we change **X =** whether contradiction detection is enabled (off vs on), what happens to **Y =** final answer accuracy after memory updates?

**My hypothesis:**  
I expect contradiction detection will help when _________, but may hurt when _________.

Write your hypothesis here:


## Scientific process plan
- **Question:** Does contradiction detection improve post-update factual answers?
- **Hypothesis:** you wrote it above
- **Experiment:** run the same scenarios with detector OFF vs ON
- **Measurement:** success rate, contradiction events, memory size after updates
- **Conclusion:** when should a memory system auto-replace contradictory facts?


# Part 1 — Tiny memory store (lecture starter code)
This lab now uses the same starter pattern from lecture:
- `remember(note)`
- `search(query, k, return_scores=False)`

This keeps the lab aligned with the DSPy ReAct memory examples.


In [2]:
# @title TinyMemory (lecture starter + forget)
class TinyMemory:
    def __init__(self):
        self.notes = []   # list of {"text":..., "tag":...}
        self.X = None     # embedding matrix: one row per document embedding

    def remember(self, text, tag=""):
        self.notes.append({"text": text, "tag": tag})
        emb = get_text_embedding(text)
        self.X = emb[None, :] if self.X is None else np.vstack([self.X, emb])

    def _do_search(self, query, k):
        q = get_text_embedding(query)
        sims = self.X @ q
        idx = np.argsort(-sims)[:k]
        return idx, sims

    def search(self, query, k=3, return_scores=False):
        if self.X is None:
            return [] if not return_scores else ([], [])
        idx, sims = self._do_search(query, k)
        results = [self.notes[int(i)] for i in idx]
        scores = [float(sims[int(i)]) for i in idx]
        return results if not return_scores else (results, scores)

    def forget(self, text, min_similarity=0.5):
        if self.X is None:
            return None
        idx, sims = self._do_search(text, k=1)
        if sims[idx[0]] >= min_similarity:
            forgotten = self.notes[idx[0]]
            del self.notes[idx[0]]
            self.X = np.delete(self.X, idx[0], axis=0)
            return forgotten["text"]
        return None


mem = TinyMemory()
mem.remember("User likes short bullet answers.", tag="pref")
mem.remember("Project Bluebird deadline is Feb 1.", tag="project")
mem.remember("Team uses GitHub issues for tasks.", tag="workflow")
print("before forget:", len(mem.notes))
_ = mem.forget("Bluebird deadline")
print("after forget:", len(mem.notes))


before forget: 3
after forget: 2


# Part 2 — Add contradiction detection + a ReAct agent
In lecture, we used **DSPy ReAct** with memory tools.

You will keep that pattern here.

You’ll implement:
- a `dspy.Signature` for contradiction classification
- `is_contradiction(memory_a, memory_b)`
- `find_contradiction(notes, new_note)`
- `remember_with_detector(memory, new_note, use_detector=True)`
- ReAct tools: `add_memory`, `retrieve_memory`, `forget_memory`


--- **TODO**

In [ ]:
# @title ✅ TODO: Contradiction detector helpers (DSPy Predict)
class MemoryContradiction(dspy.Signature):
    """Classify whether two memories contradict each other."""
    memory_a: str = dspy.InputField()
    memory_b: str = dspy.InputField()
    contradicts: bool = dspy.OutputField(desc="True if the memories contradict, else False.")


lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)

contradiction_predict = dspy.Predict(MemoryContradiction)


def is_contradiction(memory_a: str, memory_b: str) -> bool:
    """Return True if DSPy predicts these two memories contradict."""
    # TODO: call contradiction_predict(...) and return bool(pred.contradicts)
    raise NotImplementedError("Implement is_contradiction(memory_a, memory_b)")


def find_contradiction(notes, new_note: str):
    """Return index of first contradictory note, or None."""
    # TODO: iterate through notes and call is_contradiction(existing, new_note)
    raise NotImplementedError("Implement find_contradiction(notes, new_note)")


def remember_with_detector(memory: TinyMemory, new_note: str, use_detector: bool = True):
    """Store memory and optionally remove contradictory prior memory.

    Suggested policy:
      - if detector ON and contradiction found:
          1) call memory.forget(old_note_text)
          2) memory.remember(new_note)
          3) return {'contradiction': True, 'replaced_note': old_note_text}
      - otherwise store normally
    """
    # TODO: implement policy above
    raise NotImplementedError("Implement remember_with_detector(...)")


In [ ]:
# @title ReAct tool helpers (provided)
def format_notes(notes):
    if not notes:
        return "(no memory found)"
    return "\n".join([f"- {n['text']}" for n in notes])


def make_tools(memory: TinyMemory, stats: dict, use_detector: bool = True):
    """Create ReAct tools bound to this memory object."""

    def add_memory(memory_text: str) -> str:
        """Add a memory note from the user turn."""
        result = remember_with_detector(memory, memory_text, use_detector=use_detector)
        stats["writes"] = stats.get("writes", 0) + 1
        if result["contradiction"]:
            stats["contradiction_events"] = stats.get("contradiction_events", 0) + 1
            return f"stored memory; replaced contradictory note: {result['replaced_note']}"
        return "stored memory"

    def retrieve_memory(question: str) -> str:
        """Retrieve top-k memory notes relevant to a question."""
        notes = memory.search(question, k=3)
        stats["retrieves"] = stats.get("retrieves", 0) + 1
        return format_notes(notes)

    def forget_memory(memory_text: str) -> str:
        """Forget one memory by semantic match."""
        removed = memory.forget(memory_text)
        stats["forgets"] = stats.get("forgets", 0) + 1
        if removed is None:
            return "no matching memory found"
        return f"forgot: {removed}"

    return [add_memory, retrieve_memory, forget_memory]


In [ ]:
# @title ✅ Build the ReAct agent
def make_react_agent(memory: TinyMemory, stats: dict, use_detector: bool = True):
    """Return a dspy.ReAct agent wired to memory tools."""
    tools = make_tools(memory, stats, use_detector=use_detector)
    agent = dspy.ReAct(
        signature="question -> answer",
        tools=tools,
        max_iters=5,
    )
    return agent


# Part 3 — Experiment: detector OFF vs ON
Now compare two agents on the same scripted conversations:
- **Baseline:** no contradiction detector
- **Guarded:** contradiction detector ON

Goal: measure whether contradiction handling improves final factual accuracy.


In [3]:
# @title Contradiction-focused test cases
scenarios = [
    {
        "name": "single update one project",
        "turns": [
            "Remember that Project Bluebird deadline is March 12.",
            "Remember that Project Bluebird deadline is April 12.",
            "When is Bluebird due?",
        ],
        "gold_contains": "April 12",
    },
    {
        "name": "multiple updates same project",
        "turns": [
            "Remember that Project Bluebird deadline is March 12.",
            "Remember that Project Bluebird deadline is April 12.",
            "Remember that Project Bluebird deadline is May 2.",
            "When is Bluebird due?",
        ],
        "gold_contains": "May 2",
    },
    {
        "name": "two projects no cross-contamination",
        "turns": [
            "Remember that Project Bluebird deadline is April 12.",
            "Remember that Project Sparrow deadline is June 1.",
            "When is Bluebird due?",
        ],
        "gold_contains": "April 12",
    },
    {
        "name": "non-deadline memory should not break detector",
        "turns": [
            "Remember that I like bullet points.",
            "Remember that Project Bluebird deadline is July 9.",
            "When is Bluebird due?",
        ],
        "gold_contains": "July 9",
    },
]


def scenario_success(gold_contains, answer_text):
    return gold_contains.lower() in (answer_text or "").lower()


scenarios


[{'name': 'single update one project',
  'turns': ['Remember that Project Bluebird deadline is March 12.',
   'Remember that Project Bluebird deadline is April 12.',
   'When is Bluebird due?'],
  'gold_contains': 'April 12'},
 {'name': 'multiple updates same project',
  'turns': ['Remember that Project Bluebird deadline is March 12.',
   'Remember that Project Bluebird deadline is April 12.',
   'Remember that Project Bluebird deadline is May 2.',
   'When is Bluebird due?'],
  'gold_contains': 'May 2'},
 {'name': 'two projects no cross-contamination',
  'turns': ['Remember that Project Bluebird deadline is April 12.',
   'Remember that Project Sparrow deadline is June 1.',
   'When is Bluebird due?'],
  'gold_contains': 'April 12'},
 {'name': 'non-deadline memory should not break detector',
  'turns': ['Remember that I like bullet points.',
   'Remember that Project Bluebird deadline is July 9.',
   'When is Bluebird due?'],
  'gold_contains': 'July 9'}]

In [ ]:
# @title ✅ TODO: Run detector OFF vs ON experiment (ReAct)
def run_scenarios(use_detector: bool):
    rows = []
    for s in scenarios:
        memory = TinyMemory()
        stats = {"writes": 0, "retrieves": 0, "forgets": 0, "contradiction_events": 0}
        agent = make_react_agent(memory, stats, use_detector=use_detector)

        last_answer = ""
        for turn in s["turns"]:
            out = agent(question=turn)
            last_answer = getattr(out, "answer", str(out))

        rows.append({
            "scenario": s["name"],
            "detector_on": use_detector,
            "final_answer": last_answer,
            "gold_contains": s["gold_contains"],
            "success": scenario_success(s["gold_contains"], last_answer),
            "contradiction_events": stats["contradiction_events"],
            "writes": stats["writes"],
            "retrieves": stats["retrieves"],
            "forgets": stats["forgets"],
            "final_memory_size": len(memory.notes),
        })

    return pd.DataFrame(rows)


df_off = run_scenarios(use_detector=False)
df_on  = run_scenarios(use_detector=True)

print("Success rate (detector OFF):", df_off["success"].mean())
print("Success rate (detector ON): ", df_on["success"].mean())

print("\nDetector OFF")
display(df_off)
print("\nDetector ON")
display(df_on)


--- **TODO**

### Reflection
- Which scenarios improved with contradiction detection?
- Did any scenario get worse with the detector?
- What policy did your detector use to decide "contradiction"?

Write here:


---


# Part 4 (Optional) — Gradio demo
Show your contradiction-aware **ReAct** memory agent:
- toggle contradiction detector on/off
- submit one turn at a time
- inspect memory stats and answer


In [ ]:
# @title Optional: Gradio demo (minimal)
# !pip -q install gradio
import gradio as gr

ui_memory = TinyMemory()
ui_stats = {"writes": 0, "retrieves": 0, "forgets": 0, "contradiction_events": 0}
ui_agent = None
ui_detector_on = None


def ui_run(user_turn, detector_on):
    global ui_agent, ui_detector_on

    if (ui_agent is None) or (ui_detector_on != detector_on):
        ui_agent = make_react_agent(ui_memory, ui_stats, use_detector=detector_on)
        ui_detector_on = detector_on

    out = ui_agent(question=user_turn)
    answer = getattr(out, "answer", str(out))
    stats_text = f"writes={ui_stats['writes']}, retrieves={ui_stats['retrieves']}, forgets={ui_stats['forgets']}, contradiction_events={ui_stats['contradiction_events']}"
    notes_text = format_notes(ui_memory.notes)
    return stats_text, notes_text, answer


def ui_reset_memory():
    global ui_memory, ui_stats, ui_agent, ui_detector_on
    ui_memory = TinyMemory()
    ui_stats = {"writes": 0, "retrieves": 0, "forgets": 0, "contradiction_events": 0}
    ui_agent = None
    ui_detector_on = None
    return "Memory reset ✅"


demo = gr.Interface(
    fn=ui_run,
    inputs=[
        gr.Textbox(label="User turn", value="Remember that Project Bluebird deadline is March 12."),
        gr.Checkbox(label="Contradiction detector enabled", value=True),
    ],
    outputs=[
        gr.Textbox(label="Tool stats"),
        gr.Textbox(label="Current memory notes"),
        gr.Textbox(label="Answer"),
    ],
    title="Lab 8: Contradiction-aware ReAct Memory Agent",
    description="Submit turns one at a time. Toggle detector to compare behavior.",
    allow_flagging="never",
)

demo.launch(debug=False, share=False)


--- **TODO**


---

## Results
Summarize what you observed:
- success rate with detector OFF vs ON
- one scenario where contradiction handling helped
- one failure mode that still remained

Write here:


--- **TODO**

## Conclusion
- Was your hypothesis supported?
- When should contradictory memories be replaced vs kept?
- What guardrail would you add beyond contradiction detection?

Write here:


---

## 🧠 AI Usage Log

> Use this section to document any generative AI assistance (e.g., ChatGPT, Claude, Copilot) you used while completing this lab or assignment.  
> Be specific — transparency and reflection matter more than the amount of AI use.


| Tool Used | Purpose | Prompt / Context | Verification & Edits |
|------------|----------|------------------|----------------------|
| (e.g., ChatGPT (GPT-5)) | (e.g., debugging, code explanation, idea generation) | (e.g., "Why does my cosine similarity return NaN?") | (e.g., ran tests on sample input, compared with lecture code) |
| (Add rows as needed) | | | |

**Summary (2–3 sentences):**  
Briefly describe what you learned or how AI helped you think through the problem.  
Example: *AI helped me notice an off-by-one error in my indexing. I double-checked by printing intermediate results and confirmed the fix.*

---



In [ ]:
# @title ✅ Checks for Lab 8 (contradiction detector + ReAct)
print("Running checks...")

try:
    assert callable(contradiction_predict)
    print("✅ contradiction_predict configured.")
except Exception as e:
    print("❌ contradiction_predict setup failed:", e)

try:
    b = is_contradiction("Bluebird is due March 12.", "Bluebird is due April 12.")
    assert isinstance(b, bool)
    print("✅ is_contradiction returns bool.")
except Exception as e:
    print("❌ is_contradiction check failed:", e)

try:
    notes = [
        {"text": "Project Bluebird deadline is March 12.", "tag": ""},
        {"text": "Project Sparrow deadline is June 1.", "tag": ""},
    ]
    idx = find_contradiction(notes, "Project Bluebird deadline is April 12.")
    assert (idx is None) or isinstance(idx, int)
    print("✅ find_contradiction output format ok.")
except Exception as e:
    print("❌ find_contradiction check failed:", e)

try:
    m = TinyMemory()
    m.remember("Project Bluebird deadline is March 12.")
    removed = m.forget("Bluebird deadline")
    assert (removed is None) or isinstance(removed, str)
    print("✅ TinyMemory.forget output format ok.")
except Exception as e:
    print("❌ TinyMemory.forget check failed:", e)

try:
    m = TinyMemory()
    out = remember_with_detector(m, "Project Bluebird deadline is March 12.", use_detector=True)
    assert isinstance(out, dict)
    assert "contradiction" in out and "replaced_note" in out
    print("✅ remember_with_detector output format ok.")
except Exception as e:
    print("❌ remember_with_detector check failed:", e)

try:
    m = TinyMemory()
    stats = {"writes": 0, "retrieves": 0, "forgets": 0, "contradiction_events": 0}
    tools = make_tools(m, stats, use_detector=False)
    assert isinstance(tools, list) and len(tools) == 3
    print("✅ make_tools output format ok.")
except Exception as e:
    print("❌ make_tools check failed:", e)

try:
    m = TinyMemory()
    stats = {"writes": 0, "retrieves": 0, "forgets": 0, "contradiction_events": 0}
    agent = make_react_agent(m, stats, use_detector=False)
    out = agent(question="Remember that Project Bluebird deadline is March 12.")
    assert hasattr(out, "answer")
    print("✅ make_react_agent basic run ok.")
except Exception as e:
    print("❌ make_react_agent check failed:", e)

print("Done.")
